# Predictive Anayltics: Support Vector Machines

TODO: add embedded, check why there are Nans in lanlong

Approach for SVM:
– Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
– How good is your model? Evaluate your model’s performance and comment on its shortfalls.
– Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
– How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

In [46]:
DO_GRID_SEARCH = True
GRID_SAMPLE = 2000
SPATIAL_UNIT = "h3_cell" # options: census_tract, h3_cell, community_area
SPATIAL_ENCODING = "embedding" # options: embedding, latlong, onehot

In [47]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib as plt
import datetime
from sklearn.svm import SVC 
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from joblib import load, dump
import h3

import geopandas as gpd
from shapely.geometry import Polygon
from srai.neighbourhoods import H3Neighbourhood
from srai.loaders import OSMOnlineLoader
from srai.regionalizers import H3Regionalizer, geocode_to_region_gdf
from srai.joiners import IntersectionJoiner
from srai.h3 import ring_buffer_h3_regions_gdf

from sklearn.preprocessing import OneHotEncoder

## Preparations

In [48]:
# "Settings" / Decisions for the training data

# "Settings" / Decisions for the training data
if SPATIAL_UNIT == "census_tract":    
    DATA_PATH_TRAIN = "../data/train_test_data/svm_census_train.parquet"
    DATA_PATH_VAL = "../data/train_test_data/svm_census_val.parquet"
    DATA_PATH_TEST = "../data/train_test_data/svm_census_test.parquet" 
elif SPATIAL_UNIT == "h3_cell":
    DATA_PATH_TRAIN = "../data/train_test_data/svm_hexa_train.parquet"
    DATA_PATH_VAL = "../data/train_test_data/svm_hexa_val.parquet"
    DATA_PATH_TEST = "../data/train_test_data/svm_hexa_test.parquet" 
elif SPATIAL_UNIT == "community_area": 
    DATA_PATH_TRAIN = "../data/train_test_data/svm_community_train.parquet"
    DATA_PATH_VAL = "../data/train_test_data/svm_community_val.parquet"
    DATA_PATH_TEST = "../data/train_test_data/svm_community_test.parquet" 
else:
    print("Warning: No type of Spatial Unit given, Used census tract")
    DATA_PATH_TRAIN = "../data/train_test_data/svm_census_train.parquet"
    DATA_PATH_VAL = "../data/train_test_data/svm_census_val.parquet"
    DATA_PATH_TEST = "../data/train_test_data/svm_census_test.parquet" 


MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_demand"
EXCLUDE_COLS = [
    TARGET_COL,
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "trip_count",
    "date",
    "h3_cell"
]

Load data and select features and target

In [49]:
# Load data
train = pl.scan_parquet(DATA_PATH_TRAIN)
val = pl.scan_parquet(DATA_PATH_VAL)
test = pl.scan_parquet(DATA_PATH_TEST)

In [50]:
print(train.collect_schema().names())

['datetime_hour', 'month', 'weekday', 'hour', 'month_sin', 'month_cos', 'weekday_sin', 'weekday_cos', 'hour_sin', 'hour_cos', 'tmpc', 'relh', 'sknt', 'vsby', 'p01m', 'skyc1_BKN', 'skyc1_CLR', 'skyc1_FEW', 'skyc1_OVC', 'skyc1_SCT', 'skyc1_VV ', 'date', 'is_holiday', 'h3_cell', 'food_drink', 'landmark', 'shop', 'train_station', 'trip_count', 'trip_seconds_sum', 'trip_seconds_mean', 'trip_seconds_min', 'trip_seconds_max', 'trip_miles_sum', 'trip_miles_mean', 'trip_miles_min', 'trip_miles_max', 'fare_sum', 'fare_mean', 'fare_min', 'fare_max', 'tips_sum', 'tips_mean', 'tips_min', 'tips_max', 'tolls_sum', 'tolls_mean', 'tolls_min', 'tolls_max', 'extras_sum', 'extras_mean', 'extras_min', 'extras_max', 'trip_total_sum', 'trip_total_mean', 'trip_total_min', 'trip_total_max', 'most_common_payment_type']


In [51]:
keep_cols = [
    SPATIAL_UNIT,  # "h3_cell"
    "trip_count",  # target is derived from this
    "month_sin", "month_cos", "weekday_sin", "weekday_cos",
    "hour_sin", "hour_cos",
    "tmpc", "relh", "sknt", "vsby", "p01m",
    "skyc1_BKN", "skyc1_CLR", "skyc1_FEW", "skyc1_OVC", "skyc1_SCT", "skyc1_VV ",
    "is_holiday",
    "food_drink", "landmark", "shop", "train_station",
]

train_df = train.select(keep_cols).collect().to_pandas()
val_df = val.select(keep_cols).collect().to_pandas()
test_df = test.select(keep_cols).collect().to_pandas()

In [52]:
train_df.head()


,h3_cell,trip_count,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,tmpc,relh,...,skyc1_CLR,skyc1_FEW,skyc1_OVC,skyc1_SCT,skyc1_VV,is_holiday,food_drink,landmark,shop,train_station
0,882664cc05fffff,0,0.866025,0.5,0.433884,-0.900969,0.258819,0.965926,3.89,50.14,...,0,0,0,0,0,0,0.0,1.0,2.0,2.0
1,882664cc05fffff,0,0.866025,0.5,-0.781831,0.623490,0.965926,0.258819,3.33,56.71,...,0,0,0,0,0,0,0.0,1.0,2.0,2.0
2,882664cc05fffff,0,0.866025,0.5,-0.781831,0.623490,0.500000,0.866025,2.22,63.98,...,0,1,0,0,0,0,0.0,1.0,2.0,2.0
3,882664cc05fffff,0,0.866025,0.5,-0.781831,0.623490,-0.258819,-0.965926,8.61,55.84,...,0,0,0,0,0,0,0.0,1.0,2.0,2.0
4,882664cc05fffff,0,0.866025,0.5,0.433884,-0.900969,0.707107,0.707107,3.33,64.24,...,0,1,0,0,0,0,0.0,1.0,2.0,2.0


In [53]:
train_df.isna().sum()

h3_cell                0
trip_count             0
month_sin              0
month_cos              0
weekday_sin            0
weekday_cos            0
hour_sin               0
hour_cos               0
tmpc                   0
relh                   0
sknt                   0
vsby                   0
p01m                   0
skyc1_BKN              0
skyc1_CLR              0
skyc1_FEW              0
skyc1_OVC              0
skyc1_SCT              0
skyc1_VV               0
is_holiday             0
food_drink       2732215
landmark         2732215
shop             2732215
train_station    2732215
dtype: int64

In [54]:
train_df["food_drink"] = train_df["food_drink"].fillna(0.0)
train_df["landmark"] = train_df["landmark"].fillna(0.0)
train_df["shop"] = train_df["shop"].fillna(0.0)
train_df["train_station"] = train_df["train_station"].fillna(0.0)

In [55]:
# prepare data
# calculate median to split in low/high demand
# when trip_count above 50 percent use "high", when below or equal to 50 percent low


# definition for demand cause currently the q1 is 0, q2 is 1 and q3 is 3
# I excluded the zeros since a magority of values are zero
train_p90 = np.percentile(train_df.loc[train_df["trip_count"] > 0, "trip_count"], 90)
train_p70 = np.percentile(train_df.loc[train_df["trip_count"] > 0, "trip_count"], 70)
train_p50 = np.percentile(train_df.loc[train_df["trip_count"] > 0, "trip_count"], 50)
train_p25 = np.percentile(train_df.loc[train_df["trip_count"] > 0, "trip_count"], 25)

train_df["trip_demand"] = 0
train_df.loc[train_df["trip_count"] >= train_p25, "trip_demand"] = 1
train_df.loc[train_df["trip_count"] >= train_p50, "trip_demand"] = 2
train_df.loc[train_df["trip_count"] >= train_p70, "trip_demand"] = 3
train_df.loc[train_df["trip_count"] >= train_p90, "trip_demand"] = 4

val_p90 = np.percentile(val_df.loc[val_df["trip_count"] > 0, "trip_count"], 90)
val_p70 = np.percentile(val_df.loc[val_df["trip_count"] > 0, "trip_count"], 70)
val_p50 = np.percentile(val_df.loc[val_df["trip_count"] > 0, "trip_count"], 50)
val_p25 = np.percentile(val_df.loc[val_df["trip_count"] > 0, "trip_count"], 25)

val_df["trip_demand"] = 0
val_df.loc[val_df["trip_count"] >= val_p25, "trip_demand"] = 1
val_df.loc[val_df["trip_count"] >= val_p50, "trip_demand"] = 2
val_df.loc[val_df["trip_count"] >= val_p70, "trip_demand"] = 3
val_df.loc[val_df["trip_count"] >= val_p90, "trip_demand"] = 4

test_p90 = np.percentile(test_df.loc[test_df["trip_count"] > 0, "trip_count"], 90)
test_p70 = np.percentile(test_df.loc[test_df["trip_count"] > 0, "trip_count"], 70)
test_p50 = np.percentile(test_df.loc[test_df["trip_count"] > 0, "trip_count"], 50)
test_p25 = np.percentile(test_df.loc[test_df["trip_count"] > 0, "trip_count"], 25)

test_df["trip_demand"] = 0
test_df.loc[test_df["trip_count"] >= test_p25, "trip_demand"] = 1
test_df.loc[test_df["trip_count"] >= test_p50, "trip_demand"] = 2
test_df.loc[test_df["trip_count"] >= test_p70, "trip_demand"] = 3
test_df.loc[test_df["trip_count"] >= test_p90, "trip_demand"] = 4


In [56]:
print(test_df.loc[test_df["trip_count"] == 0, "trip_count"].count())
print(test_df.loc[ test_df["trip_count"] > 0, "trip_count"].count())

2446310
171782


In [57]:
model = SVC()

In [58]:
test_df.loc[test_df["trip_demand"] != "Low", "trip_demand"]

0          0
1          0
2          0
3          0
4          0
          ..
2618087    0
2618088    0
2618089    0
2618090    0
2618091    0
Name: trip_demand, Length: 2618092, dtype: int64

In [ ]:
# encoding
feature_cols = [
    col for col in train_df.columns
    if col not in EXCLUDE_COLS
]

if (SPATIAL_ENCODING == "latlong") & (SPATIAL_UNIT == "h3_cell"):
    print("Encoding: latlong and Unit: hexa")
    for df in (train_df, val_df, test_df):
        df["lat"], df["lon"] = zip(*df[SPATIAL_UNIT].map(h3.cell_to_latlng))
    


    X_train = train_df[feature_cols]
    X_val = val_df[feature_cols]
    X_test = test_df[feature_cols]
    


    # create 
    train_df_grid = train_df.sample(n=GRID_SAMPLE, random_state=42)
    X_train_grid = train_df_grid[feature_cols]


elif (SPATIAL_ENCODING == "onehot") & (SPATIAL_UNIT == "community_area"):
    # Community_area is a categorical id, not a numeric quantity, so one-hot encode it
    X_train = pd.get_dummies(train_df[feature_cols], columns=["community_area"])
    X_val = pd.get_dummies(val_df[feature_cols], columns=["community_area"])
    X_test = pd.get_dummies(test_df[feature_cols], columns=["community_area"])
    # Keep the dummy columns before scaling turns X_train into a plain array
    train_columns = X_train.columns

    # Make sure val/test have the same dummy columns as train (in case a community_area is missing)
    X_val = X_val.reindex(columns=train_columns, fill_value=0)
    X_test = X_test.reindex(columns=train_columns, fill_value=0)

    train_df_grid = train_df.sample(n=GRID_SAMPLE, random_state=42)
    X_train_grid = pd.get_dummies(train_df_grid[feature_cols], columns=["community_area"])
    X_train_grid = X_train_grid.reindex(columns=train_columns, fill_value=0)

elif SPATIAL_ENCODING == "embedding":
    # Get all unique H3 cells from your data
    all_cells = set(train_df["h3_cell"]).union(val_df["h3_cell"]).union(test_df["h3_cell"])

    # Build regions_gdf: GeoDataFrame with H3 cell polygons
    regions = []
    for cell in all_cells:
        boundary = h3.cell_to_boundary(cell)
        # h3 returns (lat, lon) tuples, shapely needs (lon, lat)
        polygon = Polygon([(lon, lat) for lat, lon in boundary])
        regions.append({"h3_cell": cell, "geometry": polygon})

    regions_gdf = gpd.GeoDataFrame(regions).set_index("h3_cell")

    # Build features_gdf: features per H3 cell (use POI counts from your data)
    # Take the mean of POI features per cell (they should be constant per cell anyway)
    poi_cols = ["food_drink", "landmark", "shop", "train_station"]
    features_gdf = train_df.groupby("h3_cell")[poi_cols].mean()

    # Fit the embedder
    neighbourhood = H3Neighbourhood()
    embedder = Hex2VecEmbedder(encoder_sizes=[64, 32])
    embeddings = embedder.fit_transform(
        regions_gdf,
        features_gdf,
        neighbourhood
    )
    # embeddings is a DataFrame indexed by h3_cell with 32 columns

    # Merge embeddings into train/val/test
    emb_cols = [f"emb_{i}" for i in range(embeddings.shape[1])]
    embeddings.columns = emb_cols

    train_df = train_df.merge(embeddings, left_on="h3_cell", right_index=True, how="left")
    val_df = val_df.merge(embeddings, left_on="h3_cell", right_index=True, how="left")
    test_df = test_df.merge(embeddings, left_on="h3_cell", right_index=True, how="left")

# creating y
y_train = train_df[TARGET_COL]
y_val = val_df[TARGET_COL]
y_test = test_df[TARGET_COL]
y_train_grid = train_df_grid[TARGET_COL]

TypeError: Hex2VecEmbedder.fit_transform() missing 1 required positional argument: 'neighbourhood'

In [62]:
help(H3Neighbourhood.__init__)

Help on function __init__ in module srai.neighbourhoods.h3_neighbourhood:

__init__(
    self,
    regions_gdf: Optional[geopandas.geodataframe.GeoDataFrame] = None,
    include_center: bool = False
) -> None
    Initializes the H3Neighbourhood.

    If a regions GeoDataFrame is provided, only the neighbours
    that are in the regions GeoDataFrame will be returned by the methods of this instance.
    NOTICE: If a region is a part of the k-th ring of a region
        and is included in the GeoDataFrame, it will be returned
        by get_neighbours_at_distance method with distance k
        even when there is no path of length k between the two regions.

    Args:
        regions_gdf (Optional[gpd.GeoDataFrame], optional): The regions that are being analyzed.
            The H3Neighbourhood will only look for neighbours among these regions.
            Defaults to None.
        include_center (bool): Whether to include the region itself in the neighbours.
        This is the default va

In [ ]:
feature_cols

['month_sin',
 'month_cos',
 'weekday_sin',
 'weekday_cos',
 'hour_sin',
 'hour_cos',
 'tmpc',
 'relh',
 'sknt',
 'vsby',
 'p01m',
 'skyc1_BKN',
 'skyc1_CLR',
 'skyc1_FEW',
 'skyc1_OVC',
 'skyc1_SCT',
 'skyc1_VV ',
 'is_holiday',
 'food_drink',
 'landmark',
 'shop',
 'train_station']

In [ ]:
print(X_train)

             month_sin  month_cos  weekday_sin  weekday_cos  hour_sin  \
0         8.660254e-01   0.500000     0.433884    -0.900969  0.258819   
1         8.660254e-01   0.500000    -0.781831     0.623490  0.965926   
2         8.660254e-01   0.500000    -0.781831     0.623490  0.500000   
3         8.660254e-01   0.500000    -0.781831     0.623490 -0.258819   
4         8.660254e-01   0.500000     0.433884    -0.900969  0.707107   
...                ...        ...          ...          ...       ...   
12207916  1.224647e-16  -1.000000     0.974928    -0.222521  0.965926   
12207917  5.000000e-01  -0.866025    -0.433884    -0.900969  0.866025   
12207918  1.224647e-16  -1.000000     0.974928    -0.222521 -0.500000   
12207919  1.224647e-16  -1.000000     0.433884    -0.900969 -0.965926   
12207920  1.224647e-16  -1.000000     0.781831     0.623490 -0.866025   

          hour_cos       tmpc       relh       sknt  vsby  ...  skyc1_CLR  \
0         0.965926   3.890000  50.140000   4.0

In [ ]:
# scaling since, SVR is distance-based, so all features need to be on a similar scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

X_train_grid = scaler.fit_transform(X_train_grid)

In [ ]:
param_grid_linear = {
    "C": [0.1, 1, 10],
    "kernel": ["linear"]
}

param_grid_rbf_sigmoid = {
    "C": [0.1, 1, 10],
    "kernel": ["rbf", "sigmoid"],
    "gamma": ["scale", "auto", 0.01, 0.1, 1]
}

param_grid_poly = {
    "C": [0.1, 1, 10],
    "kernel": ["poly"],
    "degree": [3, 4],
    "gamma": ["scale", "auto", 0.01, 0.1, 1]
}

grids = {}
for name, grid in [("linear", param_grid_linear), ("rbf_sigmoid", param_grid_rbf_sigmoid), ("poly", param_grid_poly)]:
    search = GridSearchCV(
        estimator=SVC(),
        param_grid=grid,
        cv=3,
        scoring="accuracy",
        n_jobs=-1,
        error_score="raise"
    )
    search.fit(X_train_grid, y_train_grid)
    grids[name] = search
    print(name, "best score:", search.best_score_, "best params:", search.best_params_)

best_name = max(grids, key=lambda name: grids[name].best_score_)
grid_search = grids[best_name]
print("Overall best:", best_name, grid_search.best_params_)

linear best score: 0.9339999669834752 best params: {'C': 0.1, 'kernel': 'linear'}
rbf_sigmoid best score: 0.9339999669834752 best params: {'C': 1, 'gamma': 0.01, 'kernel': 'rbf'}
poly best score: 0.9335002168585377 best params: {'C': 0.1, 'degree': 3, 'gamma': 0.01, 'kernel': 'poly'}
Overall best: linear {'C': 0.1, 'kernel': 'linear'}


In [ ]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'C': 0.1, 'kernel': 'linear'}
Best CV score: 0.9339999669834752


In [ ]:
# Evaluate on test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

#rint("Test accuracy:", accuracy_score(y_test, y_pred))

ValueError: Input X contains NaN.
SVC does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
# Train SVC 
best_model.fit(X_train, y_train)

: 

: 

In [ ]:
# Make prediction 
y_pred = grid_search.predict(X_test)

In [ ]:
# Evaluate Model
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

[[18605   471    96  5921  2521]
 [  959 93086  1727  8517    35]
 [ 1356 30912  1159 12668    23]
 [ 6039 14662   874 24101   129]
 [ 2158     1     1    12 11219]]
              precision    recall  f1-score   support

        High       0.64      0.67      0.66     27614
         Low       0.67      0.89      0.76    104324
         Mid       0.30      0.03      0.05     46118
    Mid High       0.47      0.53      0.50     45805
   Very High       0.81      0.84      0.82     13391

    accuracy                           0.62    237252
   macro avg       0.58      0.59      0.56    237252
weighted avg       0.56      0.62      0.56    237252



In [ ]:
# save model
dump(best_model, "../models/test/model_" + SPATIAL_UNIT + "_svc_" + SPATIAL_ENCODING + ".joblib")
dump(grid_search, "../models/test/grid_" + SPATIAL_UNIT + "_svc_"+ SPATIAL_ENCODING + ".joblib")

['../models/test/grid_community_svc.joblib']